# EVA-02 Base - model reference

Weights download automatically from Hugging Face Hub on first run and are cached locally afterward.

This notebook documents the model architecture: block structure, token counts, pooling behaviour, etc. used in other parts of the project. It feeds nothing downstream - `03_behavioral_export_pipeline.ipynb` and `06_nsd_export_pipeline.ipynb` each derive everything they need (data config, feature dimensionality) directly from the loaded model rather than importing anything from here. This notebook was created to inspect the model architecture for the subject Principles of Deep Learning (Principi dubokog ucenja), which was the basis for further project development of Statistical Programming. 

In [1]:
import timm
import torch

In [2]:
import logging
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)  # silences the "unauthenticated requests" notice -- harmless, just a rate-limit tip

# Load the pretrained EVA-02 Base model with its 1000-class classification head
model_name = "eva02_base_patch14_448.mim_in22k_ft_in22k_in1k"
model = timm.create_model(model_name, pretrained=True)
model.eval()

Eva(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(14, 14), stride=(14, 14))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (rope): RotaryEmbeddingCat()
  (norm_pre): Identity()
  (blocks): ModuleList(
    (0-11): 12 x EvaBlock(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
      (attn): EvaAttention(
        (q_proj): Linear(in_features=768, out_features=768, bias=True)
        (k_proj): Linear(in_features=768, out_features=768, bias=False)
        (v_proj): Linear(in_features=768, out_features=768, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
      (mlp): SwiGLU(
    

**Model architecture:**<br>
- 12-block ViT-style network,<br>
- SwiGLU-gated MLPs (not plain GELU),<br>
- rotary position embeddings (RoPE),<br>
- and a 14x14-pixel patch embedding into a 768-d token space.<br>

This printout is the reference for the block count and component names referred to elsewhere in the project.

In [3]:
# Build the preprocessing pipeline that matches what this model expects
data_config = timm.data.resolve_model_data_config(model)
transform = timm.data.create_transform(**data_config, is_training=False)

The checkpoint expects a 448 × 448, three-channel input. The model stays on CPU in
these notebooks; CUDA availability alone does not move it to a GPU.


In [4]:
# Confirm the model loaded correctly
input_size = data_config["input_size"]
print(f"Model loaded: {model_name}")
print(f"Expected input size : {input_size[1]}x{input_size[2]} px  (channels: {input_size[0]})")
print(f"Number of output classes: {model.num_classes}")
print("Model device:", next(model.parameters()).device)
print("Parameters including classification head:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Absolute position parameter present:", getattr(model, "pos_embed", None) is not None)


Model loaded: eva02_base_patch14_448.mim_in22k_ft_in22k_in1k
Expected input size : 448x448 px  (channels: 3)
Number of output classes: 1000
Model device: cpu
Parameters including classification head: 87,117,544
Absolute position parameter present: True


In [5]:
print(model.global_pool)

avg


**How does the model turn many patch tokens into one feature vector?** `avg` means simple mean-pooling across tokens, not just taking the CLS token as the summary.

In [6]:
print(model.num_prefix_tokens)   # 1 means CLS is treated as a prefix (typically excluded from avg pool)

1


**By returning 1** `num_prefix_tokens = 1` confirms the single CLS token is treated as a prefix and excluded from the average-pool checked earlier. Put together: the 768-d pre-logits vector is the mean of the 1,024 patch-token embeddings, CLS excluded, followed by final normalisation.

In [7]:
dummy = torch.randn(1, 3, 448, 448)   # 1 image, 3 channels, 448×448 — matches your config

model.eval()
with torch.no_grad():
    feats  = model.forward_features(dummy)                 # expect [1, N, 768]
    pooled = model.forward_head(feats, pre_logits=True)    # expect [1, 768]

print(feats.shape, pooled.shape)

torch.Size([1, 1025, 768]) torch.Size([1, 768])


**How many tokens does a 448x448 image actually produce?** 1,025 = 1,024 patch tokens (448/14 = 32 patches per side, 32² = 1,024) plus one CLS token;<br>
`forward_head(..., pre_logits=True)` then collapses that down to the single 768-d vector used everywhere downstream.